# Tema 10 · Proyecto final: tres maneras de resolver el mismo problema
*Curso «Google Colab» · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**El problema:** clasificar tweets en español como **negativo, neutro o positivo**.

**Tu misión:** resolverlo de **tres maneras** y comparar:

1. **Clásica:** TF-IDF + regresión logística (sin GPU).
2. **Ajuste fino:** un transformador entrenado con tus datos (tema 8, con GPU).
3. **Sin entrenar (*zero-shot*):** pedirle la respuesta a un modelo de lenguaje (Qwen, tema 8).

Al final, una rúbrica te ayuda a evaluar tu trabajo. Guarda tu cuaderno en tu Drive (**Archivo → Guardar una copia en
Drive**): esa copia es tu entrega.

**Datos:** *tweet_sentiment_multilingual*, parte en español (CC BY 3.0, Cardiff NLP). Tweets reales: algunos traen
groserías.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
# La prueba automática de MiAtlas.cl define MIATLAS_PRUEBA=1 para correr con pocos datos. Tú no necesitas tocarla.
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "| prueba rápida:", PRUEBA_RAPIDA)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: el cuaderno funciona, pero más lento. En Colab: Entorno de ejecución → Cambiar tipo → GPU.")

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn

In [ ]:
from datasets import load_dataset
import pandas as pd

datos = load_dataset("mteb/tweet_sentiment_multilingual", "spanish")
ETIQUETAS = ["negativo", "neutro", "positivo"]
entrenamiento = datos["train"].to_pandas()
prueba = datos["test"].to_pandas()
for df in (entrenamiento, prueba):
    df["label"] = df["label"].astype(int)
print(len(entrenamiento), "tweets para entrenar y", len(prueba), "para probar")
entrenamiento.head()

## Parte 1 · Conoce los datos

Responde con código y una celda de texto: ¿cuántos tweets hay de cada clase?, ¿están equilibradas?, ¿qué largo tienen
(en palabras)? Haz al menos **un gráfico**. Guarda el conteo por clase en `conteo`.

In [ ]:
conteo = None   # ← por ejemplo: entrenamiento["label"].value_counts()

In [ ]:
# Comprobación
if conteo is None:
    print("Aún no lo resuelves.")
elif int(sum(conteo)) == len(entrenamiento):
    print("✅ El conteo cubre todos los tweets")
else:
    print("❌ El conteo no suma el total de tweets")

## Parte 2 · Manera clásica: TF-IDF + regresión logística

Te dejamos el esqueleto. Completa `modelo_clasico` (un `Pipeline` de scikit-learn con `TfidfVectorizer` y
`LogisticRegression`), entrénalo y guarda la exactitud en prueba en `exactitud_clasica`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

modelo_clasico = None      # ← Pipeline([("tfidf", TfidfVectorizer(...)), ("clf", LogisticRegression(max_iter=1000))])
exactitud_clasica = None   # ← entrena con entrenamiento["text"] y mide en prueba["text"]

In [ ]:
# Comprobación
if exactitud_clasica is None:
    print("Aún no lo resuelves.")
elif 0.34 < exactitud_clasica <= 1:
    print(f"✅ Exactitud clásica: {exactitud_clasica:.1%} (al azar sería cerca de 33%)")
else:
    print("❌ Revisa: la exactitud debería ser mejor que el azar")

## Parte 3 · Ajuste fino de un transformador (con GPU)

Reutiliza el código de la **Parte B del tema 8** (BETO, 2 a 4 épocas). Guarda la exactitud en prueba
en `exactitud_ajuste`. Anota cuánto tardó y si usaste GPU.

In [ ]:
exactitud_ajuste = None   # ← tu código (puedes copiar y adaptar el del tema 8)

In [ ]:
# Comprobación
if exactitud_ajuste is None:
    print("Aún no lo resuelves.")
else:
    print(f"✅ Exactitud con ajuste fino: {exactitud_ajuste:.1%}")

## Parte 4 · Sin entrenar: preguntarle a un modelo de lenguaje

Con `pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")`, escribe una instrucción que pida responder
**solo** «negativo», «neutro» o «positivo». Clasifica al menos **100 tweets** de prueba (es lento: no todos), convierte
cada respuesta a una etiqueta (si no responde una de las tres, cuéntala como error) y guarda la exactitud en
`exactitud_llm`.

In [ ]:
exactitud_llm = None   # ← tu código

In [ ]:
# Comprobación
if exactitud_llm is None:
    print("Aún no lo resuelves.")
else:
    print(f"✅ Exactitud sin entrenar: {exactitud_llm:.1%}")

## Parte 5 · Compara y concluye

Arma una tabla con las tres exactitudes y escribe tus conclusiones en una celda de texto: ¿cuál ganó?, ¿cuánto costó
cada una (tiempo, GPU, esfuerzo)?, ¿en qué casos usarías cada manera?, ¿qué errores comete cada modelo (mira algunos
tweets mal clasificados)?

In [ ]:
resultados = {"Clásica (TF-IDF)": exactitud_clasica, "Ajuste fino": exactitud_ajuste, "Sin entrenar (LLM)": exactitud_llm}
pd.DataFrame({"exactitud": resultados}).style.format("{:.1%}", na_rep="pendiente")

## Rúbrica de autoevaluación (0 a 4 por criterio)

| Criterio | 0 | 2 | 4 |
|---|---|---|---|
| **1. Datos** | No se exploran | Conteo y un gráfico | Conteo, largo, equilibrio de clases y lo que eso implica para el resto |
| **2. Clásica** | No corre | Corre y da la exactitud | Además, el reporte por clase y una mejora probada (n-gramas, `class_weight`…) |
| **3. Ajuste fino** | No corre | Entrena y da la exactitud | Además, compara épocas o tasas de aprendizaje y reporta el tiempo con y sin GPU |
| **4. Sin entrenar** | No corre | 100 tweets clasificados | Además, prueba dos instrucciones distintas y explica la diferencia |
| **5. Comparación** | Sin conclusiones | Tabla y una conclusión | Tabla, errores típicos de cada modelo y recomendación justificada |
| **6. Cuaderno** | Solo corre en tu sesión | Corre con «Reiniciar y ejecutar todas» | Además: versiones fijas, sin claves, `CARPETA` única y explicado para otro lector |

**Total:** 24 puntos. Con 16 o más, dominas lo esencial del curso.

**Fuentes:** [scikit-learn · texto con TF-IDF](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) · [Transformers · ajuste fino](https://huggingface.co/docs/transformers/training) · [tweet_sentiment_multilingual](https://huggingface.co/datasets/mteb/tweet_sentiment_multilingual) · [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)